In [ ]:
# %%
"""
Notebook: CV Suitability Model with Responsible AI Analysis
Description:
    This notebook is designed to run in Azure Machine Learning Studio. It performs the following steps:
      1. Loads and registers a CSV dataset of CVs.
      2. Pre‑processes textual, categorical and numeric features.
      3. Trains a logistic‑regression classifier to predict whether a candidate will be accepted (historical decision).
      4. Logs metrics and registers the model with MLflow.
      5. Generates a Responsible AI (RAI) dashboard (explanations, error analysis, counterfactuals).
      6. Uploads the RAI dashboard so it appears in the Azure ML Studio portal under the model’s “Responsible AI” tab.

    The code blocks are formatted as Jupyter "# %%" cells so you can run them sequentially in the Studio notebook editor or convert the script to .ipynb with the Azure ML UI.
"""

# %%
# Install any missing dependencies (uncomment on first run)
# !pip install --upgrade azure-ai-ml azure-identity pandas scikit-learn mlflow raiwidgets azureml-responsibleai tqdm

# %%
# 1️⃣  Connect to your Azure ML workspace
import os
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Set these environment variables beforehand or replace with literal strings
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
workspace_name = os.getenv("AZURE_WORKSPACE_NAME")

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)
print(f"Connected to workspace: {ml_client.workspaces.get(workspace_name).name}")

# %%
# 2️⃣  Load the CV dataset
import pandas as pd

csv_path = "cv_dataset.csv"  # ➡️ Upload the CSV with this name to the notebook root or change the path

df = pd.read_csv(csv_path)
df.head()

# %%
# 3️⃣  (Optional) Register the dataset as an Azure ML Data asset
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

cv_data_asset = Data(
    path=csv_path,
    type=AssetTypes.URI_FILE,
    description="Dataset of CVs with historical hiring decisions",
    name="cv_suitability_dataset",
    version="1",
)
ml_client.data.create_or_update(cv_data_asset)
print("Data asset registered: cv_suitability_dataset:1")

# %%
# 4️⃣  Pre‑processing, feature engineering and model pipeline
a_target = "decision_historica"  # Binary label: Aceptado/ Rechazado
df[a_target] = df[a_target].map({"Aceptado": 1, "Rechazado": 0})

text_feature = "texto_cv"
cat_features = ["puesto_solicitado", "universidad_origen"]
num_features = ["años_experiencia"]

X = df[[text_feature] + cat_features + num_features]
y = df[a_target]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=3000, stop_words="spanish"), text_feature),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)

log_reg = LogisticRegression(max_iter=200, class_weight="balanced")

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", log_reg),
])

# %%
# 5️⃣  Train the model
pipeline.fit(X_train, y_train)

# %%
# 6️⃣  Evaluate and print metrics
from sklearn.metrics import classification_report, roc_auc_score

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
roc_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
print(f"ROC AUC: {roc_auc:.3f}")

# %%
# 7️⃣  Log to MLflow and register the model
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("cv_suitability")
with mlflow.start_run() as run:
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name="cv_suitability_logreg",
        signature=infer_signature(X_test, y_pred),
        input_example=X_test.head(),
    )
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
print(f"Model registered at: {model_uri}")

# %%
# 8️⃣  Generate Responsible AI insights
from raiutils.models import wrap_model
from responsibleai import RAIInsights
from raiwidgets import ResponsibleAIDashboard

wrapped_model = wrap_model(pipeline, model_type="classification")
rai = RAIInsights(
    model=wrapped_model,
    train=X_train,
    test=X_test,
    target_column=a_target,
    task_type="classification",
    categorical_features=cat_features,
)
rai.explainer.add()
rai.error_analysis.add()
rai.counterfactual.add(total_CFs=20)
rai.save("./rai_dashboard")
print("RAI insights generated and saved to ./rai_dashboard")

# %%
# 9️⃣  Upload the RAI dashboard so it appears in Studio
from azure.ai.ml.entities import ResponsibleAIDashboard

# Register dashboard folder as a data asset
rai_asset = ml_client.data.create_or_update(
    Data(
        name="cv_suitability_rai_dashboard",
        path="./rai_dashboard",
        type=AssetTypes.URI_FOLDER,
        description="Responsible AI dashboard for CV suitability model",
        version="1",
    )
)

# Link the dashboard to the model in the portal
rai_dash_entity = ResponsibleAIDashboard(
    name="cv_suitability_dashboard",
    title="CV Suitability Responsible AI Dashboard",
    asset_path=rai_asset.path,
    model_id="azureml:cv_suitability_logreg:1",
)
ml_client.responsibleai.create_or_update(rai_dash_entity)
print("Dashboard uploaded. Open the registered model in Studio and select the Responsible AI tab.")

# %%
# 🔟  (Optional) Render the dashboard inline for interactive exploration
ResponsibleAIDashboard(rai)
